In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, average_precision_score,accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# Data processing
def load_adj(adj_path):
    adj_sparse = sp.load_npz(adj_path).tocoo()
    adj_sparse.setdiag(0)
    adj_sparse.eliminate_zeros()
    adj = adj_sparse.toarray()
    return adj, adj.shape[0]

def load_features(feature_path):
    adata = sc.read_h5ad(feature_path)
    X = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X
    return np.nan_to_num(X, nan=0.0)

# Split edges into train/validation/test sets
def split_edges(adj, val_ratio=0.05, test_ratio=0.1, seed=42):
    pos_edges = np.argwhere(np.triu(adj == 1, k=1))
    neg_edges_candidates = np.argwhere(np.triu(adj == 0, k=1))
    print(f"[split_edges] Total positive edges: {len(pos_edges)}")
    
    pos_train_val, pos_test = train_test_split(pos_edges, test_size=test_ratio, random_state=seed)
    pos_train, pos_val = train_test_split(pos_train_val, test_size=val_ratio/(1-test_ratio), random_state=seed)
    
    np.random.seed(seed)
    neg_val_idx = np.random.choice(len(neg_edges_candidates), size=len(pos_val), replace=False)
    neg_val = neg_edges_candidates[neg_val_idx]
    neg_edges_candidates = np.delete(neg_edges_candidates, neg_val_idx, axis=0)
    neg_test_idx = np.random.choice(len(neg_edges_candidates), size=len(pos_test), replace=False)
    neg_test = neg_edges_candidates[neg_test_idx]

    train_pos = torch.from_numpy(pos_train).long()
    val_pos = torch.from_numpy(pos_val).long()
    val_neg = torch.from_numpy(neg_val).long()
    test_pos = torch.from_numpy(pos_test).long()
    test_neg = torch.from_numpy(neg_test).long()

    return train_pos, val_pos, val_neg, test_pos, test_neg


class AE(nn.Module):
    def __init__(self, n_input, n_enc1, n_enc2, n_enc3, n_z, dropout=0.2):
        super().__init__()
        self.enc1 = nn.Linear(n_input, n_enc1)
        self.enc2 = nn.Linear(n_enc1, n_enc2)
        self.enc3 = nn.Linear(n_enc2, n_enc3)
        self.z_layer = nn.Linear(n_enc3, n_z)    # z
        self.dec3 = nn.Linear(n_z, n_enc3)
        self.dec2 = nn.Linear(n_enc3, n_enc2)
        self.dec1 = nn.Linear(n_enc2, n_enc1)
        self.dec0 = nn.Linear(n_enc1, n_input)     # x-bar
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h1 = F.relu(self.enc1(x)) 
        h1 = self.dropout(h1)
        h2 = F.relu(self.enc2(h1))
        h2 = self.dropout(h2)      
        h3 = F.relu(self.enc3(h2)) 
        h3 = self.dropout(h3)
        z = self.z_layer(h3) 
        return h1, h2, h3, z, self.decode(z)

    def decode(self, z):
        d3 = F.relu(self.dec3(z))
        d2 = F.relu(self.dec2(d3))
        d1 = F.relu(self.dec1(d2))
        return self.dec0(d1)     # x_bar

class FeatureClusterAE(nn.Module):
    def __init__(self, input_dim, h1, h2, h3, z_dim, n_clusters, v=1.0):
        super().__init__()
        self.ae = AE(input_dim, h1, h2, h3, z_dim)
        self.cluster_layer = nn.Parameter(torch.Tensor(n_clusters, z_dim))
        nn.init.xavier_normal_(self.cluster_layer.data)
        self.v = v

    def forward(self, x):
        _, _, _, z, x_bar = self.ae(x)
        distances = torch.sum((z.unsqueeze(1) - self.cluster_layer)**2, dim=2)
        q = 1.0 / (1.0 + distances/self.v)
        q = q.pow((self.v+1.0)/2.0)
        q = (q.t()/torch.sum(q,1)).t()
        return x_bar, q, z

def target_distribution(q):
    weight = (q**2) / torch.sum(q, dim=0)
    return (weight.t()/torch.sum(weight, dim=1)).t()

class reconstruction_graph(nn.Module):
    def __init__(self, NE, temperature=0.1):
        super().__init__()
        # Set as a learnable parameter
        self.nodes_embedding = nn.Parameter(NE)
        self.temperature = temperature

    def forward(self, CC):
        cluster_similarity = torch.mm(self.nodes_embedding, CC.t())
        soft_assign = F.softmax(cluster_similarity/self.temperature, dim=1)
        # Cluster-based similarity
        cluster_based_sim = torch.mm(soft_assign, soft_assign.t())
        # Direct similarity
        node_norm = F.normalize(self.nodes_embedding, p=2, dim=1)
        direct_sim = (torch.mm(node_norm, node_norm.t())+1)/2
        # Fuse the two similarity matrices
        fused_sim = 0.7*direct_sim + 0.3*cluster_based_sim
        return fused_sim

class update_nodes_embedding(nn.Module):
    def __init__(self, ne, temperature=0.1):
        super().__init__()
        self.reconstruction_module = reconstruction_graph(ne, temperature)
        self.optimizer = torch.optim.SGD(self.reconstruction_module.parameters(), lr=1e-4, momentum=0.9)
        self.loss_fn = nn.MSELoss(reduction='sum')

    def forward(self, adj, CC, edge_train_idx):
        self.reconstruction_module.train()
        for _ in range(1):
            self.optimizer.zero_grad()
            graph_recons = self.reconstruction_module(CC)
            preds = graph_recons[edge_train_idx[:,0], edge_train_idx[:,1]]
            labels = adj[edge_train_idx[:,0], edge_train_idx[:,1]]
            loss = self.loss_fn(preds, labels)
            loss.backward()
            self.optimizer.step()
        return self.reconstruction_module.nodes_embedding.detach()

def pretrain_ae(model, x, device, epochs=100, lr=1e-3):
    model.train()
    optimizer = torch.optim.Adam(model.ae.parameters(), lr=lr)
    for epoch in range(epochs):
        optimizer.zero_grad()
        _, _, _, _, x_bar = model.ae(x)
        loss = F.mse_loss(x_bar, x)
        loss.backward()
        optimizer.step()
        if (epoch+1)%10==0:
            print(f"pretrain_AE Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")


def train(adj_path, feature_path, n_clusters=12, n_z=50, n_epochs=500, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Processed adjacency matrix and number of nodes
    adj_np, num_nodes = load_adj(adj_path)
    adj = torch.FloatTensor(adj_np).to(device)
    # Load gene expression matrix
    x_np = load_features(feature_path)
    x = torch.FloatTensor(x_np).to(device)

    train_pos, val_pos, val_neg, test_pos, test_neg = split_edges(adj_np, val_ratio=0.05, test_ratio=0.1)
    train_pos = train_pos.to(device)
    val_pos, val_neg = val_pos.to(device), val_neg.to(device)
    test_pos, test_neg = test_pos.to(device), test_neg.to(device)

    model = FeatureClusterAE(x.shape[1], 128, 256, 128, n_z, n_clusters).to(device)
    print(f"[Train] Input feature dimension (x.shape[1]): {x.shape[1]}")
    pretrain_ae(model, x, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Initialize cluster centers with KMeans
    cluster_centers = KMeans(n_clusters=n_clusters, n_init=20).fit(x.cpu().numpy()).cluster_centers_
    model.cluster_layer.data = torch.tensor(cluster_centers).float().to(device)

    ne = x.detach().to(device)
    updater = update_nodes_embedding(ne.clone().detach()).to(device)

    auc_list, ap_list = [], []

    for epoch in range(n_epochs):
        model.train()
        
        # Update cluster centers (CC) while keeping node embeddings (NE) fixed
        x_bar, q, _ = model(ne)
        p = target_distribution(q)
        kl_loss = F.kl_div(q.log(), p, reduction='batchmean')
        recon_loss = F.mse_loss(x_bar, x)
    
        # Update node embeddings (NE) while keeping cluster centers (CC) fixed
        ne_fixed = ne.detach().clone() 
        ne_updated = updater(adj, model.cluster_layer.data, train_pos).to(device)
        cons_loss = F.mse_loss(ne_updated, ne_fixed)
        ne = ne_updated.detach().clone() 
    
        loss = kl_loss + 0.1 * recon_loss + 0.1 * cons_loss
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Validate every 50 epochs
        if epoch%50==0 or epoch==n_epochs-1:
            recon = reconstruction_graph(ne).to(device)
            recon.eval()
            with torch.no_grad():
                pred_adj = recon(model.cluster_layer.data)
                pred_scores = (pred_adj - pred_adj.min())/(pred_adj.max()-pred_adj.min()+1e-9)
                
                val_edges = torch.cat([val_pos, val_neg], dim=0)
                val_labels = torch.cat([torch.ones(len(val_pos)), torch.zeros(len(val_neg))]).to(device)
                val_preds = pred_scores[val_edges[:,0], val_edges[:,1]]
                val_ap = average_precision_score(val_labels.cpu().numpy(), val_preds.cpu().numpy())
                val_auc = roc_auc_score(val_labels.cpu().numpy(), val_preds.cpu().numpy())
                val_preds_binary = (val_preds >= 0.5).long().cpu().numpy()
                val_labels_np = val_labels.cpu().numpy()

                val_acc = accuracy_score(val_labels_np, val_preds_binary)
                val_precision = precision_score(val_labels_np, val_preds_binary)
                val_recall = recall_score(val_labels_np, val_preds_binary)
                val_f1 = f1_score(val_labels_np, val_preds_binary)
                
                print(f"Epoch {epoch}: Val ACC={val_acc:.4f}, Val AUC={val_auc:.4f}, Val AP={val_ap:.4f}, "
      f"Precision={val_precision:.4f}, Recall={val_recall:.4f}, F1={val_f1:.4f}")
    
    # Test set
    recon = reconstruction_graph(ne).to(device)
    recon.eval()
    with torch.no_grad():
        pred_adj = recon(model.cluster_layer.data)
        pred_scores = (pred_adj - pred_adj.min())/(pred_adj.max()-pred_adj.min()+1e-9)
        test_edges = torch.cat([test_pos, test_neg], dim=0)
        test_labels = torch.cat([torch.ones(len(test_pos)), torch.zeros(len(test_neg))]).to(device)
        test_preds = pred_scores[test_edges[:,0], test_edges[:,1]]
        test_ap = average_precision_score(test_labels.cpu().numpy(), test_preds.cpu().numpy())
        test_auc = roc_auc_score(test_labels.cpu().numpy(), test_preds.cpu().numpy())
        test_preds_binary = (test_preds >= 0.5).long().cpu().numpy()
        test_labels_np = test_labels.cpu().numpy()

        test_acc = accuracy_score(test_labels_np, test_preds_binary)
        test_precision = precision_score(test_labels_np, test_preds_binary)
        test_recall = recall_score(test_labels_np, test_preds_binary)
        test_f1 = f1_score(test_labels_np, test_preds_binary)

        print(f"Final Test: ACC={test_acc:.4f}, AUC={test_auc:.4f}, AP={test_ap:.4f}, "
      f"Precision={test_precision:.4f}, Recall={test_recall:.4f}, F1={test_f1:.4f}")

    # Perform KMeans clustering and plot (after alternating updates)
    node_embeddings_np = ne.cpu().numpy()
    kmeans_final = KMeans(n_clusters=12, n_init=20, random_state=0).fit(node_embeddings_np)
    labels_final = kmeans_final.labels_

    from sklearn.decomposition import PCA
    emb_2d = PCA(n_components=2).fit_transform(node_embeddings_np)

    plt.figure(figsize=(6, 5))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels_final, cmap='tab20', s=10)
    plt.title("Final clustering result")
    plt.colorbar(label="Cluster")
    plt.tight_layout()
    plt.savefig("cluster_final_embedding.png", dpi=300)

    return ne.cpu(), model.cluster_layer.detach().cpu(), test_auc, test_ap

In [ ]:
# ----- 运行 -----
if __name__=="__main__":
    adj_path = "gat_adj_sparse_topk.npz"
    feature_path = "gat_embeddings.h5ad"
    node_embeddings, cluster_centers, test_auc, test_ap = train(adj_path, feature_path, n_clusters=16, n_z=50, n_epochs=500)

    pd.DataFrame(node_embeddings.numpy()).to_excel("node_embeddings.xlsx", index=False, header=None)
    pd.DataFrame(cluster_centers.numpy()).to_excel("cluster_centers.xlsx", index=False, header=None)

[split_edges] Total positive edges: 16922
[Train] Input feature dimension (x.shape[1]): 50
pretrain_AE Epoch 10/100, Loss: 0.034524
pretrain_AE Epoch 20/100, Loss: 0.034015
pretrain_AE Epoch 30/100, Loss: 0.031775
pretrain_AE Epoch 40/100, Loss: 0.028887
pretrain_AE Epoch 50/100, Loss: 0.026038
pretrain_AE Epoch 60/100, Loss: 0.023505
pretrain_AE Epoch 70/100, Loss: 0.021166
pretrain_AE Epoch 80/100, Loss: 0.019288
pretrain_AE Epoch 90/100, Loss: 0.017799
pretrain_AE Epoch 100/100, Loss: 0.016487
Epoch 0: Val ACC=0.9723, Val AUC=0.9921, Val AP=0.9885, Precision=0.9474, Recall=1.0000, F1=0.9730
Epoch 50: Val ACC=0.9693, Val AUC=0.9920, Val AP=0.9881, Precision=0.9422, Recall=1.0000, F1=0.9702
Epoch 100: Val ACC=0.9687, Val AUC=0.9920, Val AP=0.9881, Precision=0.9411, Recall=1.0000, F1=0.9697
Epoch 150: Val ACC=0.9664, Val AUC=0.9920, Val AP=0.9881, Precision=0.9369, Recall=1.0000, F1=0.9674
Epoch 200: Val ACC=0.9646, Val AUC=0.9919, Val AP=0.9881, Precision=0.9338, Recall=1.0000, F1=0.9